# Classifier transfor leaning from LLM 情緒類別分類器(遷移學習)

This is a comprahensive notebook and tutorial on how to fine tune the `qwen-0.5b` classification model

Fine-tuning Qwen-0.5B (a smaller model) with LoRA (Low-Rank Adaptation) is an efficient approach that requires less computational power.

It got a classification accuracy of 0.93.


# Transformers變形金剛(變壓器)模型

<img src="https://cdn.analyticsvidhya.com/wp-content/uploads/2019/06/Screenshot-from-2019-06-17-19-53-10.png">

Bert and GPT are parts of Transformer

<img src="https://heidloff.net/assets/img/2023/02/transformers.png" width="800">

Encoders and Decoders
As mentioned, there are encoders and decoders. BERT uses encoders only, GTP uses decoders only. Both options understand language including syntax and semantics. Especially the next generation of large language models like GPT with billions of parameters do this very well.

The two models focus on different scenarios. However, since the field of foundation models is evolving, the differentiation is often fuzzier.

BERT (encoder): classification (e.g., sentiment), questions and answers, summarization, named entity recognition
GPT (decoder): translation, generation (e.g., stories)
The outputs of the core models are different:

BERT (encoder): Embeddings representing words with attention information in a certain context
GPT (decoder): Next words with probabilities
Both models are pretrained and can be reused without intensive training. Some of them are available as open source and can be downloaded from communities like Hugging Face, others are commercial. Reuse is important, since trainings are often very resource intensive and expensive which few companies can afford.

The pretrained models can be extended and customized for different domains and specific tasks. Layers can sometimes be reused without modifications and more layers are added on top. If layers need to be modified, the new training is more expensive. The technique to customize these models is called Transfer Learning, since the same generic model can easily be transferred to other domains.

[Source](https://heidloff.net/article/foundation-models-transformers-bert-and-gpt/)

### Encoder-Decoder Transformers



<img src='https://miro.medium.com/v2/resize:fit:1400/format:webp/1*vrSX_Ku3EmGPyqF_E-2_Vg.png'>

### Attension Layer
<img src='https://miro.medium.com/v2/resize:fit:640/format:webp/1*aTLu4HxVUzEOIZTLHmOktw.png'>

# Colab GPU
## 設定執行階段 變更執行階段類型 硬體加速器  選取GPU  


In [1]:
!nvidia-smi

Sat May 24 15:27:27 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 列出GPU與CPU資源
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 14346530960944514404
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 14619377664
locality {
  bus_id: 1
  links {
  }
}
incarnation: 14489212002461153143
physical_device_desc: "device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5"
xla_global_id: 416903419
]


In [3]:
# Specify the GPU device to use
#import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## Ubuntu版本資訊

In [4]:
!lsb_release -a

No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 22.04.4 LTS
Release:	22.04
Codename:	jammy


# Insatll packages


        transformers: For model handling.
        peft: For LoRA integration.
        datasets: For dataset handling.
        accelerate: For distributed training.
        bitsandbytes: For memory-efficient training (optional but useful for Qwen).

In [5]:
#!pip install transformers peft datasets accelerate bitsandbytes

In [6]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.9 MB/s eta 0:00:00


# 掛載雲端硬碟

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 切換工作目錄到雲端硬碟目錄下(取用資料比較方便，可以用相對路徑)

你要改成你自己雲端硬碟目錄的路徑，可複製路徑並貼上，不要用鍵盤輸入!

In [8]:
# cd to_your_folder
# cd命令的前面一行不要加上說明文字，否則colab的cd會認不得指令

In [9]:
cd /content/drive/MyDrive/Colab Notebooks

/content/drive/MyDrive/Colab Notebooks


In [10]:
ls -l

total 14860
-rw------- 1 root root   312580 May 18 16:44 '「1-Training model using Transformers Trainer.ipynb」的副本'
-rw------- 1 root root    49197 May 24 15:27  「1-改尾分類微調情緒分類Qwen0.5b-v4-5epochs-acc0.93.ipynb」的副本
-rw------- 1 root root    47656 May 19 05:34 '「2-sentiment classification model usage.ipynb」的副本'
drwx------ 2 root root     4096 May 13 02:50  best-model-v1/
drwx------ 2 root root     4096 May 13 02:35  checkpoints_v1/
drwx------ 2 root root     4096 May 20 04:17  checkpoints_v4/
-rw------- 1 root root 14071136 May 13 02:33  dataset_reviews.csv
-rw------- 1 root root      324 Oct 26  2023  Untitled0.ipynb
-rw------- 1 root root    11527 Feb 13 06:56  Untitled10.ipynb
-rw------- 1 root root    49223 Mar  9 15:14  Untitled11.ipynb
-rw------- 1 root root     4504 Mar  3 02:12  Untitled12.ipynb
-rw------- 1 root root      258 Mar  9 14:59  Untitled13.ipynb
-rw------- 1 root root    91763 Mar 10 03:23  Untitled14.ipynb
-rw------- 1 root root    38925 Mar 24 01:59  Untitled15.ipynb
-

In [11]:
import torch
import datasets
import pandas as pd
import evaluate
import numpy as np

# Load Huggingface transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
from transformers import TrainingArguments, Trainer
from transformers import BertTokenizer, BertTokenizerFast, BertForSequenceClassification
from sklearn import metrics
import torch
import evaluate
import datasets


In [12]:
# Setting device on GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


# Read data

簡體中文資料，資料集來自於網路，轉換成繁體中文

In [13]:
df = pd.read_csv('./dataset_reviews.csv', sep='|')

In [14]:
df

,text,label
0,做父母一定要有劉墉這樣的心態，不斷地學習，不斷地進步，不斷地給自己補充新鮮血液，讓自己保持一...,1
1,作者真有英國人嚴謹的風格，提出觀點、進行論述論證，儘管本人對物理學瞭解不深，但是仍然能感受到...,1
2,作者長篇大論借用詳細報告數據處理工作和計算結果支持其新觀點。為什麼荷蘭曾經縣有歐洲最高的生產...,1
3,作者在戰幾時之前用了〞擁抱〞令人叫絕．日本如果沒有戰敗，就有會有美軍的佔領，沒胡官僚主義的延...,1
4,作者在少年時即喜閱讀，能看出他精讀了無數經典，因而他有一個龐大的內心世界。他的作品最難能可貴...,1
...,...,...
80437,以前幾乎天天吃，現在調料什麼都不放，,0
80438,昨天訂涼皮兩份，什麼調料都沒有放，就放了點麻油，特別難吃，丟了一份，再也不想吃了,0
80439,"涼皮太辣,吃不下都",0
80440,本來遲到了還自己點！！！,0


In [15]:
df.dtypes

,0
text,object
label,int64


# Convert the format of y

Convert the format of y from int to LongTensor

## Convert label using one-hot representation 輸出資料格式one-hot轉換
    
    轉成用2個節點表達兩類
    類別0: [1 0]  
    類別1: [0 1]  


    如果是3個類別用3個節點表之:
    類別0: [1 0 0]  
    類別1: [0 1 0]  
    類別2: [0 0 1]

負面情緒Negative 0 --> [1 0]

正面情緒Positive 1 --> [0 1]

## Easy Represention 0,1,2... (內部會自動轉換為one-hot)

    負面情緒Negative 0 --> [0]

    正面情緒Positive 1 --> [1]

    如果是3個類別
    中立情緒 Neutral 2 --> [2]


In [16]:
# Map labels to integers
categories=['負面','正面']

In [17]:

label_to_id = { cate : i for i, cate in enumerate(categories)}

In [18]:
label_to_id

{'負面': 0, '正面': 1}

In [19]:
id_to_label = { i : cate for i, cate in enumerate(categories)}

In [20]:
id_to_label

{0: '負面', 1: '正面'}

# Sample some examples for demonstration

In [21]:
df = df.sample(10000)

# Conver pandas dataframe to Huggingface Dataset

In [22]:
dataset = datasets.Dataset.from_pandas(df, preserve_index=False)
# eval_data = Dataset.from_pandas(X_eval)

In [23]:
dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 10000
})

In [24]:
dataset[0]

{'text': '當時活動價買的挺便宜，真心不知道是不是假貨！！洗了一段時間發現頭髮好油好油！！！真的好油！！！一開始以為是護髮素用多了，後來發現不用護髮素都油的不行，自從用了惠潤的就這樣！現在頭髮1天洗一次都油的不行，真心覺得是假貨！！',
 'label': 0}

In [25]:
dataset.to_pandas().head(5)

,text,label
0,當時活動價買的挺便宜，真心不知道是不是假貨！！洗了一段時間發現頭髮好油好油！！！真的好油！！...,0
1,房間很小，與行政間的名字不符。另外這家酒店的按摩可千萬不能叫，超級黑！結帳時居然會有打手上來...,0
2,服務很差，房間沒有窗戶，告訴我「不保證提供有窗戶的房間」，所以安排住進了一個像洞一樣的房間不...,0
3,第二次住，在北京，398元的房間，已經很不錯了。沒處吃早飯，酒店早飯要100多，吃不起。單人...,1
4,"當當就因為這個盤的退換貨,讓我徹底認識到當當只買,不退換,至少不真心退換,讓我們瞭解到當當不...",0


# Load Tokenizer

In [26]:

# model_id = "google/gemma-3-1b"
# model_id = "Qwen/Qwen2.5-0.5B"
model_id = "Qwen/Qwen2.5-0.5B-instruct"

In [27]:
tokenizer = AutoTokenizer.from_pretrained(model_id)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [28]:
# 必須在Huggingface註冊，取得API token才能下載模型
#access_token = "???"
#tokenizer = AutoTokenizer.from_pretrained(model_id, token=access_token)

# Tokenzie text

In [29]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        #padding="max_length",
        max_length=512,
        truncation=True,
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [30]:
tokenized_dataset

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 10000
})

In [31]:
# tokenized_dataset[0]

# Split dataset for training and testing

# Split dataset: Train, Test (Val)

Training set: 訓練資料集 -->給模型讀進去訓練

Test set: 測試資料集 -->驗證或測試模型的準確度



Split dataset into 90% for training and 10% for testing

In [32]:
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.05, seed=1234)
train_data = tokenized_dataset["train"]
test_data = tokenized_dataset["test"]

In [33]:
train_data

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 9500
})

In [34]:
print(test_data)

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 500
})


In [35]:
train_data[0]

{'text': '賓館服務非常好，環境也很好，但在賓館吃飯太貴了，地角很偏，交通很不方便。',
 'label': 1,
 'input_ids': [117993,
  110057,
  100831,
  108014,
  3837,
  108131,
  74763,
  101243,
  3837,
  104965,
  117993,
  110057,
  99405,
  103373,
  99222,
  108871,
  34187,
  3837,
  29490,
  63836,
  99165,
  99835,
  3837,
  99735,
  99165,
  113897,
  1773],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1]}

#  定義Model

這裡的做法有簡單的版本也有複雜的版本

簡單版:


        model = AutoModelForSequenceClassification.from_pretrained(
            model_id,  # 模型名稱
            num_labels=len(categories),  # Number of output labels
        )


複雜版:




In [36]:
from transformers import Qwen2Model, Trainer, TrainingArguments, AutoTokenizer, AutoModel
from torch import nn
from transformers.modeling_outputs import SequenceClassifierOutput
import os

In [37]:
import torch
import os
import torch.nn.functional as F
from torch import nn

class QwenForClassifier(nn.Module):
    def __init__(self, base_model, hidden_size, num_labels):
        super(QwenForClassifier, self).__init__()
        # 凍結 base model 的參數
        self.base_model = base_model

        for param in self.base_model.parameters():
            param.requires_grad = False

        # 注意力池化機制
        self.attention_pooler = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        # 多層融合權重 (最後4層)
        self.layer_weights = nn.Parameter(torch.ones(4) / 4)

        # 增強型分類器
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.2),

            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1),

            nn.Linear(128, num_labels)
        )

        # 保存配置
        self.config = base_model.config
        self.config.num_labels = num_labels

    def forward(self, input_ids, attention_mask=None, labels=None):
        # 獲取所有隱藏層狀態
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )

        # 獲取最後4層隱藏狀態
        hidden_states = outputs.hidden_states
        if hidden_states is None:
            # 如果模型沒有返回hidden_states，使用last_hidden_state
            last_hidden = outputs.last_hidden_state
            sequence_output = last_hidden
        else:
            # 融合最後4層 (或可用層數)
            last_layers = hidden_states[-4:] if len(hidden_states) >= 4 else hidden_states[1:]
            layer_weights = F.softmax(self.layer_weights[:len(last_layers)], dim=0)

            # 加權融合多層特徵
            sequence_output = torch.zeros_like(last_layers[0])
            for i, layer in enumerate(last_layers):
                sequence_output += layer_weights[i].unsqueeze(-1).unsqueeze(-1) * layer

        # 注意力池化
        attention_scores = self.attention_pooler(sequence_output)
        attention_probs = F.softmax(attention_scores, dim=1)
        context_vector = torch.matmul(attention_probs.transpose(-1, -2), sequence_output).squeeze(1)

        # 也計算平均池化向量
        mean_pooled = torch.mean(sequence_output, dim=1)

        # 結合注意力池化和平均池化 (殘差連接)
        combined_repr = context_vector + mean_pooled

        # 分類預測
        logits = self.classifier(combined_repr)

        # 計算損失
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)

        return {"loss": loss, "logits": logits}

    def save_model(self, output_dir=None):
        """保存分類器權重和配置"""
        os.makedirs(output_dir, exist_ok=True)

        # 保存分類器權重
        classifier_path = os.path.join(output_dir, "classifier_weights.pt")
        model_dict = {
            'classifier': self.classifier.state_dict(),
            'attention_pooler': self.attention_pooler.state_dict(),
            'layer_weights': self.layer_weights,
            'config': {
                'num_labels': self.config.num_labels,
                'hidden_size': self.config.hidden_size
            }
        }
        torch.save(model_dict, classifier_path)
        print(f"已保存分類器權重至 {classifier_path}")

    def load_model(self, model_dir, device=None):
        """載入分類器權重"""
        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        classifier_path = os.path.join(model_dir, "classifier_weights.pt")
        if os.path.exists(classifier_path):
            model_dict = torch.load(classifier_path, map_location=device, weights_only=True)

            # 載入各組件
            self.classifier.load_state_dict(model_dict['classifier'])
            self.attention_pooler.load_state_dict(model_dict['attention_pooler'])
            self.layer_weights.data = model_dict['layer_weights'].to(device)

            print(f"已載入分類器權重: {classifier_path}")
            return True
        else:
            print(f"警告: 找不到分類器權重檔案 {classifier_path}")
            return False

## 初始化模型


        分類任務：兩者都可用，但 AutoModel 更輕量
        生成功能：只有 AutoModelForCausalLM 支持
        內存使用：AutoModelForCausalLM 通常較大，因為包含了完整的語言模型頭
        在 Qwen2 情感分類模型中，使用 AutoModelForCausalLM 更為靈活，因為它既可以進行分類，也保留了原始的文本生成功能。



        # AutoModelForCausalLM 輸出
        outputs = base_model(input_ids, attention_mask)
        # 輸出包含：
        # - loss: (可選) 語言模型損失
        # - logits: 張量，形狀為 [batch_size, sequence_length, vocab_size]
        # - past_key_values: (可選) 用於加速解碼的過去狀態
        # - hidden_states: (可選) 所有隱藏層狀態的元組
        # - attentions: (可選) 注意力權重的元組


        # AutoModel 輸出
        outputs = base_model(input_ids, attention_mask)
        # 輸出包含：
        # - last_hidden_state: 張量，形狀為 [batch_size, sequence_length, hidden_size]
        # - hidden_states: (可選) 所有隱藏層狀態的元組
        # - attentions: (可選) 注意力權重的元組

In [38]:
len(categories)

2

In [39]:
# 在外部先載入base_model預訓練權重(不包含分類層)
full_model = AutoModelForCausalLM.from_pretrained(model_id)
#
hidden_size = full_model.config.hidden_size
model = QwenForClassifier(full_model.model, hidden_size, num_labels= len(categories))

# 移動到指定設備
model = model.to(device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [40]:
full_model.model

Qwen2Model(
  (embed_tokens): Embedding(151936, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2Attention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
)

In [41]:
full_model.base_model

Qwen2Model(
  (embed_tokens): Embedding(151936, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2Attention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
)

## 看看base_model與model有何不同?

model內部有: 一個base_model+輸出分類層。它被拼接為分類器，可以做兩個類別的分類

base_model仍舊是GPT自回歸模型

但是在記憶體，兩者是共享同一份base_model權重，model有多一個分類器的輸出層的部分

In [42]:
full_model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2RotaryEmbe

In [43]:
model

QwenForClassifier(
  (base_model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2Rota

In [44]:
# 這與model是一樣的模型??
model.base_model

Qwen2Model(
  (embed_tokens): Embedding(151936, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2Attention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
)

# 模型怎麼用?

尚未訓練的模型 (classifier的數據都是0)

In [45]:

# Function to make predictions
def predict_sentiment(text, model, tokenizer, device):
    # Tokenize the input text
    inputs = tokenizer(
        text,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # Get model predictions
    with torch.no_grad():
        outputs = model(**inputs)

    # Extract logits and apply softmax to get probabilities
    # logits = outputs.logits
    logits = outputs["logits"]  # 取出 logits


    probabilities = torch.nn.functional.softmax(logits, dim=-1)

    # Get the predicted class (0: negative, 1: positive)
    predicted_class = torch.argmax(probabilities, dim=-1).item()

    # Get the class name using id_to_label
    predicted_label = id_to_label[predicted_class]

    # Get the confidence score
    confidence = probabilities[0][predicted_class].item()

    return {
        "text": text,
        "sentiment": predicted_label,
        "confidence": round(confidence,2),
        "probabilities": {
            id_to_label[i]: round(prob.item(),2) for i, prob in enumerate(probabilities[0])
        }
    }


In [46]:
text = "今天天氣真好，我很開心"
predict_sentiment(text, model, tokenizer, device)

{'text': '今天天氣真好，我很開心',
 'sentiment': '負面',
 'confidence': 0.53,
 'probabilities': {'負面': 0.53, '正面': 0.47}}

In [47]:
text = "這個產品品質差，服務更糟糕"
predict_sentiment(text, model, tokenizer, device)

{'text': '這個產品品質差，服務更糟糕',
 'sentiment': '負面',
 'confidence': 0.6,
 'probabilities': {'負面': 0.6, '正面': 0.4}}

In [48]:
text = "這家餐廳的食物美味，環境也很舒適"
predict_sentiment(text, model, tokenizer, device)

{'text': '這家餐廳的食物美味，環境也很舒適',
 'sentiment': '負面',
 'confidence': 0.74,
 'probabilities': {'負面': 0.74, '正面': 0.26}}

## 模型每個參數層都是Trainable

In [49]:
def print_model_details(model):
    print("\n" + "="*80)
    print("MODEL ARCHITECTURE WITH TRAINABILITY STATUS")
    print("="*80)

    trainable_params = 0
    non_trainable_params = 0

    # Print each layer with details
    for idx, (name, layer) in enumerate(model.named_modules(), 1):
        if not list(layer.named_children()):  # Only print leaf nodes (actual layers)
            param_count = sum(p.numel() for p in layer.parameters())
            status = "TRAINABLE" if any(p.requires_grad for p in layer.parameters()) else "NON-TRAINABLE"

            print(f"\nLayer #{idx:02d}")
            print(f"├─ Name: {name}")
            print(f"├─ Type: {layer.__class__.__name__}")
            print(f"├─ Details: {layer}")
            print(f"├─ Parameters: {param_count:,}")
            print(f"└─ Status: {status}")

            if status == "TRAINABLE":
                trainable_params += param_count
            else:
                non_trainable_params += param_count

    total_params = trainable_params + non_trainable_params
    trainable_percentage = (trainable_params / total_params) * 100 if total_params > 0 else 0

    print("\n" + "="*80)
    print(f"Total Trainable Parameters: {trainable_params:,}")
    print(f"Total Non-Trainable Parameters: {non_trainable_params:,}")
    print(f"Total Parameters: {total_params:,}")
    print(f"Trainable Parameters Percentage: {trainable_percentage:.2f}%")
    print("="*80)

In [50]:
print_model_details(model)


MODEL ARCHITECTURE WITH TRAINABILITY STATUS

Layer #03
├─ Name: base_model.embed_tokens
├─ Type: Embedding
├─ Details: Embedding(151936, 896)
├─ Parameters: 136,134,656
└─ Status: NON-TRAINABLE

Layer #07
├─ Name: base_model.layers.0.self_attn.q_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=896, bias=True)
├─ Parameters: 803,712
└─ Status: NON-TRAINABLE

Layer #08
├─ Name: base_model.layers.0.self_attn.k_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=128, bias=True)
├─ Parameters: 114,816
└─ Status: NON-TRAINABLE

Layer #09
├─ Name: base_model.layers.0.self_attn.v_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=128, bias=True)
├─ Parameters: 114,816
└─ Status: NON-TRAINABLE

Layer #10
├─ Name: base_model.layers.0.self_attn.o_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=896, bias=False)
├─ Parameters: 802,816
└─ Status: NON-TRAINABLE

Layer #12
├─ Name: base_model.layers.0.mlp.gate_proj
├─ Typ

In [51]:
# 檢查原始參數是否可訓練
# print("Classifier weight requires_grad:", model.classifier.weight.requires_grad)

## 模型主體固定參數，只有極少數參數是Trainable

In [52]:
#model.print_trainable_parameters()

In [53]:
def print_trainable_parameters(model):
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    all_params = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {trainable_params:,d} || all params: {all_params:,d} || trainable%: {100 * trainable_params / all_params:.2f}%")

In [54]:
print_trainable_parameters(model)

trainable params: 378,503 || all params: 494,411,271 || trainable%: 0.08%


In [55]:
print_model_details(model)


MODEL ARCHITECTURE WITH TRAINABILITY STATUS

Layer #03
├─ Name: base_model.embed_tokens
├─ Type: Embedding
├─ Details: Embedding(151936, 896)
├─ Parameters: 136,134,656
└─ Status: NON-TRAINABLE

Layer #07
├─ Name: base_model.layers.0.self_attn.q_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=896, bias=True)
├─ Parameters: 803,712
└─ Status: NON-TRAINABLE

Layer #08
├─ Name: base_model.layers.0.self_attn.k_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=128, bias=True)
├─ Parameters: 114,816
└─ Status: NON-TRAINABLE

Layer #09
├─ Name: base_model.layers.0.self_attn.v_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=128, bias=True)
├─ Parameters: 114,816
└─ Status: NON-TRAINABLE

Layer #10
├─ Name: base_model.layers.0.self_attn.o_proj
├─ Type: Linear
├─ Details: Linear(in_features=896, out_features=896, bias=False)
├─ Parameters: 802,816
└─ Status: NON-TRAINABLE

Layer #12
├─ Name: base_model.layers.0.mlp.gate_proj
├─ Typ

## Use EOS token as padding

In [56]:
tokenizer.pad_token

'<|endoftext|>'

In [57]:
tokenizer.eos_token

'<|im_end|>'

In [58]:
model.pad_token_id = tokenizer.eos_token_id  # Use EOS token as padding

# Train

In [59]:

class CustomTrainer(Trainer):
    def save_model(self, output_dir=None, _internal_call=False):
        """保存分類器權重和配置"""
        # 使用預設output_dir如果未指定
        output_dir = output_dir if output_dir is not None else self.args.output_dir
        os.makedirs(output_dir, exist_ok=True)

        # 保存分類器權重
        classifier_path = os.path.join(output_dir, "classifier_weights.pt")
        model_dict = {
            'classifier': self.model.classifier.state_dict(),
            'attention_pooler': self.model.attention_pooler.state_dict(),
            'layer_weights': self.model.layer_weights,
            'config': {
                'num_labels': self.model.config.num_labels,
                'hidden_size': self.model.config.hidden_size
            }
        }

        # Save model config (required by HF Trainer)
        if hasattr(self.model, "config"):
            self.model.config.save_pretrained(output_dir)

        torch.save(model_dict, classifier_path)
        print(f"已保存分類器權重至 {classifier_path}")

        # 保存訓練狀態
        super().save_state()

        return output_dir



In [60]:
#metric = evaluate.combine(["accuracy", "f1", "precision", "recall"])
#metric = evaluate.combine(["accuracy"])
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)  # Convert probabilities to predicted labels
    metric = evaluate.load('accuracy')
    return metric.compute(predictions=predictions, references=labels)

In [61]:
# requries several GB of GPU memory
training_args = TrainingArguments(
    output_dir="checkpoints_v4",  # Output directory for checkpoints
    learning_rate=5e-5,  # Learning rate for the optimizer
    weight_decay=0.01,  # Weight decay for regularization
    warmup_steps=500,
    seed=101,

    per_device_train_batch_size=26,  # Batch size per device
    per_device_eval_batch_size=3,  # Batch size per device for evaluation

    num_train_epochs=5,  # Number of training epochs

    eval_strategy='steps',  # Evaluate after each epoch
    save_strategy="steps",  # Save model checkpoints after each epoch

    #load_best_model_at_end=True,  # Load the best model based on the chosen metric
    save_total_limit=2,
    push_to_hub=False,  # Disable pushing the model to the Hugging Face Hub
    report_to="none",  # Disable logging to Weight&Bias
    #fp16=True, # 是否用此精度訓練Whether to use fp16 16-bit (mixed) precision training instead of 32-bit training.
    #bf16=True, # for 新型GPU 才能設定bf16
    logging_steps=1000,
)

In [62]:
trainer = CustomTrainer(
    model=model,  # The LoRA-adapted model
    #tokenizer=tokenizer,  # Tokenizer for the model
    processing_class=tokenizer,  # Tokenizer for the model

    train_dataset=train_data,  # Training dataset
    eval_dataset=test_data,  # Evaluation dataset
    args=training_args,  # Training arguments
    compute_metrics=compute_metrics,  # Function to calculate evaluation metrics
)

## Resume training from the last checkpoint if available

接續訓練

訓練後，手動載入之前訓練的分類器權重

In [63]:

# checkpoint_path = "./checkpoints_v3/checkpoint-3167"
# model_path = "trained_classifier_v3"
# model.load_model(checkpoint_path)

## Let's train the model

In [64]:
%%time
model.config.use_cache = False  # Disable cache for faster training
trainer.train()
# trainer.train(resume_from_checkpoint=True) # Resume training from a checkpoint
#trainer.train(resume_from_checkpoint="./checkpoints_v1/checkpoint-19002")

Step,Training Loss,Validation Loss,Accuracy
1000,0.426800,0.215279,0.920000


已保存分類器權重至 checkpoints_v4/checkpoint-500/classifier_weights.pt


已保存分類器權重至 checkpoints_v4/checkpoint-1000/classifier_weights.pt
已保存分類器權重至 checkpoints_v4/checkpoint-1500/classifier_weights.pt
已保存分類器權重至 checkpoints_v4/checkpoint-1830/classifier_weights.pt
CPU times: user 40min 22s, sys: 38.6 s, total: 41min 1s
Wall time: 41min 27s


TrainOutput(global_step=1830, training_loss=0.33586385758196724, metrics={'train_runtime': 2487.2269, 'train_samples_per_second': 19.098, 'train_steps_per_second': 0.736, 'total_flos': 0.0, 'train_loss': 0.33586385758196724, 'epoch': 5.0})

# Make a test

In [65]:
text = "今天天氣真好，我很開心"
predict_sentiment(text, model, tokenizer, device)

{'text': '今天天氣真好，我很開心',
 'sentiment': '正面',
 'confidence': 0.97,
 'probabilities': {'負面': 0.03, '正面': 0.97}}

In [66]:
text = "這個產品品質差，服務更糟糕"
predict_sentiment(text, model, tokenizer, device)

{'text': '這個產品品質差，服務更糟糕',
 'sentiment': '負面',
 'confidence': 0.95,
 'probabilities': {'負面': 0.95, '正面': 0.05}}

# Save Model

In [67]:
#存lora_model
# model_path = "trained_lora_model_v1"
# trainer.model.save_pretrained(model_path)

In [68]:
# 這樣存檔 - 只會儲存分類器權重和設定
# model_path = "trained_classifier_v4"
# trainer.save_model(model_path)
# model.save_pretrained(model_path)

In [69]:
# 這樣存檔 - 只會儲存分類器權重和設定
model_path = "trained_classifier_v4"
model.save_model(model_path)
# model.save_pretrained(model_path)

已保存分類器權重至 trained_classifier_v4/classifier_weights.pt


# Evaluate the model

In [70]:
def predict(input_text):
    inputs = tokenizer(input_text, return_tensors="pt").to(device)  # Convert to PyTorch tensors and move to GPU (if available)
    with torch.no_grad():
        #outputs = model(**inputs).logits  # Get the model's output logits
        outputs = model(**inputs)['logits']  # Get the model's output logits
        y_prob = torch.sigmoid(outputs).tolist()[0]  # Apply sigmoid activation and convert to list
    return np.round(y_prob, 2)  # Round the predicted probability to 2 decimal places

In [71]:
def inference(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs['logits'], dim=1)
    #probs = torch.softmax(outputs.logits, dim=1)
    return probs.argmax().item()

In [72]:
df_test = pd.DataFrame(data=test_data)

In [73]:
df_test

,text,label,input_ids,attention_mask
0,不想說什麼，只想祝福所有騙人不負責的京東的相關人員遭受報應。,0,"[104359, 100379, 102386, 3837, 109254, 105514,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,酸蘋果，再也不想在京東買水果了,0,"[99918, 99714, 233, 27773, 3837, 112870, 99172...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
2,一直都很信賴京東，這次太讓人失望了，618買的，理解你們訂單多，顯示620送達，一直到621...,0,"[99725, 105037, 21317, 116654, 46553, 102356, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,華聖 高原紅富士蘋果 6個裝 1.2kg 新鮮水果，新鮮好吃。,1,"[104967, 107670, 18137, 40419, 52129, 105693, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,本人於2008年6月5日入住該酒店標準間。裝修很別緻，設施也不錯，前臺服務很好。早餐品種不多...,1,"[102043, 99823, 17, 15, 15, 23, 7948, 21, 9754...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
...,...,...,...,...
495,1.房間內有很大的氣味（可能是牆紙膠的味道），簡直無法入睡！2.晚上窗外路上有拖拉機來來往往...,0,"[16, 13, 99218, 55865, 100456, 109711, 104200,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
496,攜程訂的1層房間整體條件很差，不值那個價錢，需要重新裝修,0,"[13531, 250, 38507, 108106, 9370, 16, 106953, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
497,大小不一，很多壞掉了，不值這個價錢，！！！！,0,"[92032, 16530, 14777, 3837, 99555, 114115, 105...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
498,這箱是最滿意的一箱了，大小挺均勻，就是一打開有味道，是箱子的味道不太好聞，不過馬上拿出來放冰...,1,"[99672, 73296, 104890, 101496, 36589, 99774, 7...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [74]:
input_text='''立法院今天發布人事通報，總務處長周傑調任參事室參事並代理總務處長，引發外界聯想立法院長韓國瑜出手進行人事調整。周傑今天受訪表示，正常調動，謝謝關心。
部分媒體報導，質疑立法院總務處長人事異動，是否與先前立法院立委辦公室傳出有裝潢費昂貴、不合理等情形，或是日前立法院司法及法制委員會有委員提議要更改格局，而周傑回應「建議要更動是否可在會期中研議規畫，休會時再改」，令國民黨立法院黨團總召傅崐萁不以為然有關。'''
predict(input_text)

array([0.31, 0.63])

In [75]:
input_text = "我不高興"
inference(input_text)

1

In [76]:
%%time
df_test['prediction'] = df_test['text'].map(predict)
df_test['y_pred'] = df_test['prediction'].apply(lambda x: np.argmax(x, axis=0))

#
# df_test['prediction'] = df_test['text'].map(inference)

CPU times: user 16.6 s, sys: 66.8 ms, total: 16.6 s
Wall time: 16.8 s


In [77]:
accuracy = (df_test['y_pred'] == df_test['label']).mean()
print(f"Model Accuracy on Test Data: {accuracy:.4f}")
df_test.head()

Model Accuracy on Test Data: 0.8980


,text,label,input_ids,attention_mask,prediction,y_pred
0,不想說什麼，只想祝福所有騙人不負責的京東的相關人員遭受報應。,0,"[104359, 100379, 102386, 3837, 109254, 105514,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.5, 0.44]",0
1,酸蘋果，再也不想在京東買水果了,0,"[99918, 99714, 233, 27773, 3837, 112870, 99172...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]","[0.65, 0.27]",0
2,一直都很信賴京東，這次太讓人失望了，618買的，理解你們訂單多，顯示620送達，一直到621...,0,"[99725, 105037, 21317, 116654, 46553, 102356, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.38, 0.52]",1
3,華聖 高原紅富士蘋果 6個裝 1.2kg 新鮮水果，新鮮好吃。,1,"[104967, 107670, 18137, 40419, 52129, 105693, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.12, 0.81]",1
4,本人於2008年6月5日入住該酒店標準間。裝修很別緻，設施也不錯，前臺服務很好。早餐品種不多...,1,"[102043, 99823, 17, 15, 15, 23, 7948, 21, 9754...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.07, 0.85]",1


In [78]:
cm = metrics.confusion_matrix(df_test['label'], df_test['y_pred'])
cm

array([[215,  31],
       [ 20, 234]])

In [79]:
print(metrics.classification_report(df_test['label'], df_test['y_pred']))

              precision    recall  f1-score   support

           0       0.91      0.87      0.89       246
           1       0.88      0.92      0.90       254

    accuracy                           0.90       500
   macro avg       0.90      0.90      0.90       500
weighted avg       0.90      0.90      0.90       500



In [80]:
metrics.precision_score(df_test['label'], df_test['y_pred'], average='micro')

0.898

In [81]:
metrics.precision_score(df_test['label'], df_test['y_pred'], average='macro')

0.8989562424729025

In [82]:
metrics.recall_score(df_test['label'], df_test['y_pred'], average='micro')

0.898

In [83]:
metrics.recall_score(df_test['label'], df_test['y_pred'], average='macro')

0.8976217911785418

# Load model

In [84]:
# 在外部先載入base_model預訓練權重(不包含分類層)
# full_model = AutoModelForCausalLM.from_pretrained(model_id)
#
model_path = "trained_classifier_v4"
# model_path = "checkpoints_v3\checkpoint-4145"
hidden_size = full_model.config.hidden_size
model = QwenForClassifier(full_model.model, hidden_size, num_labels= len(categories))

model.load_model(model_path, device=device)

# 移動到指定設備
model = model.to(device)

已載入分類器權重: trained_classifier_v4/classifier_weights.pt


In [85]:
text = "這個產品品質差，服務更糟糕"
predict_sentiment(text, model, tokenizer, device)

{'text': '這個產品品質差，服務更糟糕',
 'sentiment': '負面',
 'confidence': 0.93,
 'probabilities': {'負面': 0.93, '正面': 0.07}}

In [86]:
full_model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2RotaryEmbe

# Text generation fro full_model

In [87]:
from IPython.display import Markdown

In [88]:
def generate_text(input_prompt):

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": input_prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    prompt_length = model_inputs['input_ids'].shape[1]

    generated_ids = full_model.generate(
        model_inputs.input_ids,
        max_new_tokens=512,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(generated_ids[0][prompt_length:], skip_special_tokens=True)
    return response


In [89]:
text="給出三個保持健康的提示。"
result = generate_text(text)
Markdown(result)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


1. 避免過度飲食，注意攝取均衡的營養。
2. 保持良好的睡眠習慣，確保充足的休息時間。
3. 勤於運動，以提高心肺功能和體重控制能力。

In [90]:
%%time
text="我們如何減少空氣污染？請給幾項重要的建議。"
result = generate_text(text)
Markdown(result)

CPU times: user 10.4 s, sys: 36.7 ms, total: 10.5 s
Wall time: 10.5 s


要減少空气污染，可以從以下几个方面入手：

1. 减少車輛使用：車輛排放的有害气体和颗粒物是造成空气污染的主要原因之一。因此，盡量選擇公共交通工具、車購車種、電動車等低碳出行方式。

2. 設計和維修環境：設計和維修工業和工業區以減少污染源，以及提高能源效率，降低污染物排放。

3. 離散和密閉空間：避免在密閉空間中工作或生活，例如在城市中心、工廠等場所設置開放式空間。

4. 使用低能耗設備：使用高效能設備和設備來減少能源消耗和排放。

5. 支持綠色能源：支持使用可再生能源，如風力、水力、太阳能等。

6. 提倡環保生活方式：包括節約用水、減少食物浪费、減少垃圾產生、少用塑料袋等。

7. 加強監管和評估：政府應該加強對環境污染的監管和評估，並採取有效的措施來控制和減少污染。

8. 接納和接受新技術：積極接受和採納新的技術和方法，以減輕對環境的影響。

9. 個人行為：個人也應參與到保護環境的行動中去，比如節省資源、減少碳排放等。

以上是一些基本的建議，但需要根據本地的環境和情況作出調整和優化。